In [1]:
!pip install -r ../requirements.txt

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [ ]:
import os
import logging
import ast

import dotenv
dotenv.load_dotenv()


import pandas as pd
import numpy as np
import torch

from transformers import (BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments)


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, hamming_loss


from clearml import Dataset, Task

In [7]:
logging.basicConfig(level=logging.INFO)

# task = Task.init(project_name="HSE-GP5", task_name='Bert')

configuration={'max_features':5000, 'ngram_range':(1,2), 'stop_words':'english', 'model_state':42, 'test_size':0.2, 'val_size': 0.1, 'split_state':42}
# task.connect(configuration)

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [4]:
dataset = Dataset.get(dataset_project="HSE-GP5", dataset_name='final_lyrics_dataset')
dataset_path = dataset.get_local_copy()

df = pd.read_csv(os.path.join(dataset_path,'final_dataset.csv'),encoding='utf-8',escapechar="\\")
df = df[['lyrics','genres']].dropna()
df['genres'] = df['genres'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genres'])

In [11]:
# task.upload_artifact(name="genre_classes",artifact_object=set(mlb.classes_))

XtrainVal, Xtest, ytrainVal, ytest = train_test_split(df['lyrics'], y, test_size=configuration['test_size'], random_state=configuration['split_state'])
Xtrain, Xval, ytrain, yval = train_test_split(XtrainVal, ytrainVal, test_size=configuration['val_size'], random_state=configuration['split_state'])

In [8]:
tokenizer = BertTokenizerFast.from_pretrained('google-bert/bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=len(mlb.classes_), problem_type='multi_label_classification').to(device) # https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForSequenceClassification, https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google-bert/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-bert/bert-base-uncased/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google-bert/bert-base-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-bert/bert-base-uncased/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
INFO:h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
class BertDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, text, labels):
        self.embeddings = tokenizer(text=text, truncation=True, padding='max_length', return_tensors='pt') # https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PythonBackend.__call__, pytorch, обрезаем текст если не влезает
        self.labels = torch.tensor(labels, dtype=torch.float32) # переводим тензоры y в float32
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.embeddings.items()}
        item['labels'] = self.labels[idx]
        return item

In [12]:
train_dataset = BertDataset(text=Xtrain.tolist(), labels=ytrain, tokenizer=tokenizer)
val_dataset = BertDataset(text=Xval.tolist(), labels=yval, tokenizer=tokenizer)
test_dataset = BertDataset(text=Xtest.tolist(), labels=ytest, tokenizer=tokenizer)

In [ ]:
def evaluate(preds):
    logits, labels = preds
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)
    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(labels, preds)
    }

In [14]:
trainer = Trainer(model, 
                args=TrainingArguments(
                    output_dir='./bert', # куда сохранять ЧП во время обучения
                    num_train_epochs=5, # Число эпох обучения
                    learning_rate=1e-5, # lr, меньше т.к. файнтуним
                    weight_decay=0.01, # L2-регуляризация
                    per_device_train_batch_size=32, # Какого размера передаем батч на каждый шаг
                    per_device_eval_batch_size=32,
                    warmup_steps=500, # Распределение шагов до того, как модель придет к 1e-3
                    eval_strategy='epoch', # Как часто считаем метрики на валидации
                    save_strategy='epoch', # Как часто сохранять ЧП
                    load_best_model_at_end=True, # Загружать итоговую модель не последнюю, а лучшую по метрике
                    metric_for_best_model='f1_micro', # Определяем f1_micro как метрику, по которой определяем лучшую версию модели f1-micro — считаем TP, FP, FN по всем классам, а потом суммируем (взвешенный вариант f1-macro)
                    logging_steps=100 # Как часто логируем
                    ),
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                compute_metrics=evaluate
            ) # https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer

In [15]:
trainer.train()

/Users/timurmif/ACADEMY/hse/smadimo-hse/.conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

KeyboardInterrupt



In [ ]:
evaluate(trainer.predict(test_dataset))